# M1 method


## From scratch

In [10]:
import pandas as pd
import numpy as np
from util.filter_data import filtering
from methods.decision_tree_scratch import DecisionTreeScratch

df1 = filtering('src/claims_train.csv')

def kfold_split(n_samples, k=5, seed=42):
    np.random.seed(seed)
    indices = np.arange(n_samples)
    np.random.shuffle(indices)
    return np.array_split(indices, k)


def cross_validate_tree(df, target_col, tree_class, features, k=5, seed=42, **tree_kwargs):
    folds = kfold_split(len(df), k=k, seed=seed)
    mse_scores = []
    best_tree = None
    best_mse = float("inf")

    for i in range(k):
        val_idx = folds[i]
        train_idx = np.setdiff1d(np.arange(len(df)), val_idx)

        # selcts features and target column
        X_train = df.iloc[train_idx][features].to_numpy()
        y_train = df.iloc[train_idx][target_col].to_numpy()
        X_val = df.iloc[val_idx][features].to_numpy()
        y_val = df.iloc[val_idx][target_col].to_numpy()

        # trains tree using the hyperparameters
        tree = tree_class(**tree_kwargs)
        tree.fit(X_train, y_train)

        # pruining
        tree.prune(X_val, y_val)

        # Evaluate validation MSE
        y_val_pred = tree.predict(X_val)
        mse = np.mean((y_val - y_val_pred)**2)
        mse_scores.append(mse)

        # Keep the best tree
        if mse < best_mse:
            best_mse = mse
            best_tree = tree

        print(f"Fold {i+1}/{k} - Validation MSE: {mse:.4f}")

    print(f"Best validation MSE: {best_mse:.4f}")
    return mse_scores, best_tree

In [11]:
def predict_tree(tree, df, features):
    X = df[features].to_numpy()
    return tree.predict(X)


features = ["VehPower", "VehAge", "DrivAge", "BonusMalus", "Density"]

# Perform 5-fold CV and get the best tree
mse_scores, best_tree = cross_validate_tree(
    df=df1,
    target_col="Risk",
    tree_class=DecisionTreeScratch,
    features=features,
    k=5,
    max_depth=5,
    min_samples_split=20
)

# Predict a single row
single_row = df1.iloc[0:1]  # keep as DataFrame
pred_single = predict_tree(best_tree, single_row, features)
print(f"Predicted risk for single row: {pred_single[0]:.5f}")
# First 10 actual target values
actual_values = df1["Risk"].iloc[:10]
print("Actual risk values for first 10 rows:")
print(actual_values.values)


# Predict multiple rows
pred_batch = predict_tree(best_tree, df1.iloc[:10], features)
print("Predictions for first 10 rows:", pred_batch)

Fold 1/5 - Validation MSE: 0.0285
Fold 2/5 - Validation MSE: 0.0286
Fold 3/5 - Validation MSE: 0.0284
Fold 4/5 - Validation MSE: 0.0286
Fold 5/5 - Validation MSE: 0.0284
Best validation MSE: 0.0284
Predicted risk for single row: 0.21920
Actual risk values for first 10 rows:
[0.44779761 0.59454042 0.37738764 0.19304709 0.59454042 0.4276604
 0.08701661 0.08701661 0.10427201 0.0935331 ]
Predictions for first 10 rows: [0.21919506 0.23282468 0.18120818 0.26987773 0.35539316 0.21919506
 0.26987773 0.2019538  0.24901564 0.26987773]


## Using reference

# M2 method - Neural Network


## From scratch

In [12]:
from methods.neural_network_scratch import NeuralNetworkScratch
from util.filter_data import filtering
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = filtering('src/claims_train.csv', alpha=2, gamma=0.1, train=True)

Y = df['Risk'].values.astype(np.float32).reshape(-1, 1)
df_clean = df.drop(columns=["IDpol", "ClaimNb", "Exposure","Region",'Risk'])
numeric_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = df_clean.select_dtypes(include=["object", "category"]).columns

X_cat = pd.get_dummies(df_clean[categorical_cols], drop_first=True)
df_clean = df_clean.drop(columns=categorical_cols)
X_final = np.hstack([df_clean, X_cat.values])

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_final).astype(np.float32)



In [13]:
nn = NeuralNetworkScratch(
    input_dim=X_scaled.shape[1],
    hidden1_dim=128,
    hidden2_dim=64,
    output_dim=1,
    learning_rate=0.1,
    epochs=400,
    number_of_batches=4000,
    random_state=42,
    patience=5,
    learning_shrink=False
)

nn.train(X_scaled, Y)

Epoch 0, Loss: 0.02905937225952676
Validation Loss: 0.028916230786224066
Epoch 1, Loss: 0.028974627701872705
Validation Loss: 0.028838438885546495
Epoch 2, Loss: 0.02891308737893434
Validation Loss: 0.028786198409257818
Epoch 3, Loss: 0.02860681067502998
Validation Loss: 0.02849390677734417
Epoch 4, Loss: 0.02851491296174775
Validation Loss: 0.02843001703081325
Epoch 5, Loss: 0.028439443710928797
Validation Loss: 0.028377541236031232
Epoch 6, Loss: 0.028414851446573103
Validation Loss: 0.028338953878191544
Epoch 7, Loss: 0.02837737530288888
Validation Loss: 0.02832315763741414
Epoch 8, Loss: 0.02836492969178155
Validation Loss: 0.028308549354998776
Epoch 9, Loss: 0.028316374645843544
Validation Loss: 0.028269646875811967
Epoch 10, Loss: 0.028331868723192617
Validation Loss: 0.02828404262366097
Epoch 11, Loss: 0.028343857913411714
Validation Loss: 0.028294182551149058
Epoch 12, Loss: 0.028369826668657748
Validation Loss: 0.02830757611508509
Epoch 13, Loss: 0.02827207427166388
Validation

## Using reference

In [14]:
from util.filter_data import filtering
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = filtering('src/claims_train.csv', alpha=2, gamma=0.1, train=True)

Y = df['Risk'].values.astype(np.float32).reshape(-1, 1)
df_clean = df.drop(columns=["IDpol", "ClaimNb", "Exposure","Region",'Risk'])
numeric_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = df_clean.select_dtypes(include=["object", "category"]).columns

X_cat = pd.get_dummies(df_clean[categorical_cols], drop_first=True)
df_clean = df_clean.drop(columns=categorical_cols)
X_final = np.hstack([df_clean, X_cat.values])

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_final).astype(np.float32)

# Flatten Y
y_flat = Y.ravel()


In [15]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error


# 80/20 train–test split
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, Y, test_size=0.2, random_state=42
)

# Define neural network
mlp = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=50,          # number of epochs
    batch_size=100,       # mini-batch size
    random_state=42,
    verbose=True          # prints loss per iteration
)

# Train
mlp.fit(X_train, y_train)

# Predict
y_train_pred = mlp.predict(X_train)
y_val_pred   = mlp.predict(X_val)

# Compute MSE
mse_train_mlp = mean_squared_error(y_train, y_train_pred)
mse_val_mlp   = mean_squared_error(y_val,   y_val_pred)

print("MLPRegressor train MSE:", mse_train_mlp)
print("MLPRegressor val   MSE:", mse_val_mlp)

/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:1771: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Iteration 1, loss = 0.01506393
Iteration 2, loss = 0.01432664
Iteration 3, loss = 0.01423009
Iteration 4, loss = 0.01418953
Iteration 5, loss = 0.01415959
Iteration 6, loss = 0.01414674
Iteration 7, loss = 0.01413979
Iteration 8, loss = 0.01412467
Iteration 9, loss = 0.01412746
Iteration 10, loss = 0.01411882
Iteration 11, loss = 0.01411487
Iteration 12, loss = 0.01411057
Iteration 13, loss = 0.01410728
Training loss did not improve more than tol=0.000100 for 10 consecutive epochs. Stopping.
MLPRegressor train MSE: 0.028149258345365524
MLPRegressor val   MSE: 0.02815365232527256


# M3 method